# W13-D7 配套实验：最终 Virtual CTO Review 的三个数据判断

本 notebook 是《第13周-Day7-VirtualCTO-两周总复盘与集成评估.md》的可执行验证，不复述文章，只做三件事：

1. **剪刀差曲线复现**：六周两对象（LnkChat / MallSenseAI）的五维评分 vs 理解深度——验证"评分下探 = 测量精度提高"这条跨对象规律
2. **五维雷达对照**：MallSenseAI 望远镜分（W12）vs 内窥镜分（W13）vs LnkChat 终值（W11）——验证 6.3-6.8 预测区间与"信任边界按 Code Health 画"
3. **集成木桶蒙特卡洛**：MallSenseAI × LnkChat 集成就绪度 = max(视觉侧, 平台侧)——验证"集成窗口由慢的一侧决定，帮慢侧才是唯一有效加速"

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)
rng = np.random.default_rng(42)

## 实验 0：六周评分数据与综合分自算

数据来自每周日 Virtual CTO Review 的正式评分（W8-W11 评 LnkChat，W12-W13 评 MallSenseAI）。先自算综合分，验证 7.05→6.8 复评落在 W12 预测区间 6.3-6.8 上沿。

In [ ]:
DIMS = ["AQ", "CH", "ADR", "TD", "DX"]

# LnkChat W8-W11 五维评分（W11-D7 终版趋势表）
lnkchat = {
    "W8":  dict(zip(DIMS, [7.5, 7.0, 8.0, 6.5, 7.0])),
    "W9":  dict(zip(DIMS, [7.5, 6.5, 7.0, 6.0, 7.0])),
    "W10": dict(zip(DIMS, [7.5, 6.5, 7.0, 6.0, 7.0])),
    "W11": dict(zip(DIMS, [7.5, 6.0, 6.5, 5.5, 6.5])),
}
# MallSenseAI 两次评分（W12-D7 望远镜 / W13-D7 内窥镜）
mall = {
    "W12": dict(zip(DIMS, [7.0, 7.5, 6.0, 7.0, 7.5])),
    "W13": dict(zip(DIMS, [7.0, 7.0, 6.0, 6.5, 7.5])),
}
# 理解深度（每周自评）
understanding = {"W8": 7.0, "W9": 7.5, "W10": 8.5, "W11": 9.0, "W12": 8.0, "W13": 8.5}

overall = lambda d: np.mean(list(d.values()))
print(f"MallSenseAI W12 综合分 = {overall(mall['W12']):.2f}（首评记录 7.05，四舍五入一致）")
print(f"MallSenseAI W13 综合分 = {overall(mall['W13']):.2f}  → 预测区间 [6.3, 6.8] 上沿命中：{6.3 <= overall(mall['W13']) <= 6.8}")
print(f"LnkChat     W11 综合分 = {overall(lnkchat['W11']):.2f}（终版记录 6.4）")
print(f"两周下探 {overall(mall['W12']) - overall(mall['W13']):.2f} vs LnkChat 四周下探 {overall(lnkchat['W8']) - overall(lnkchat['W11']):.2f} —— 同构剪刀差")

## 实验 1：剪刀差曲线（双对象）

假设：**理解深度上升 + 五维评分下探，在两个对象上同构复现**。若成立，则"评分下探不是对象变差，是测量精度提高"可以定稿为标准评审方法论。

In [ ]:
weeks = ["W8", "W9", "W10", "W11", "W12", "W13"]
lnk_line = [overall(lnkchat[w]) for w in weeks[:4]]
mall_line = [overall(mall[w]) for w in weeks[4:]]
und_line = [understanding[w] for w in weeks]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(4), lnk_line, "o-", color="tab:blue", lw=2.2, label="LnkChat 五维综合（W8-W11）")
ax.plot(range(4, 6), mall_line, "s-", color="tab:red", lw=2.2, label="MallSenseAI 五维综合（W12-W13）")
ax.plot(range(6), und_line, "^--", color="tab:green", lw=2, ms=8, label="理解深度（自评）")
ax.axvline(3.5, color="gray", ls=":", lw=1.2)
ax.text(3.56, 6.15, "对象切换\n（平台 → 能力包）", fontsize=9, color="gray")
ax.annotate("望远镜 7.05", xy=(4, mall_line[0]), xytext=(3.2, 7.35), fontsize=9,
            arrowprops=dict(arrowstyle="->", color="tab:red", lw=1))
ax.annotate("内窥镜 6.8\n命中预测区间上沿", xy=(5, mall_line[1]), xytext=(4.35, 6.05), fontsize=9,
            arrowprops=dict(arrowstyle="->", color="tab:red", lw=1))
ax.annotate("四周下探 0.8", xy=(3, lnk_line[3]), xytext=(1.4, 6.3), fontsize=9,
            arrowprops=dict(arrowstyle="->", color="tab:blue", lw=1))
ax.set_xticks(range(6)); ax.set_xticklabels(weeks)
ax.set_ylim(5.8, 9.5); ax.set_ylabel("分数")
ax.set_title("剪刀差在两个对象上同构复现：理解深度↑ 五维评分↓", fontsize=13)
ax.legend(loc="upper left", fontsize=9); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig("/root/learning-notebooks/第13周/w13d7_scissors_two_objects.png", dpi=130)
plt.show()

# 规律检验：两个对象都是 理解深度末值>首值 且 评分末值<首值
ok = (understanding["W13"] > understanding["W12"]) and (mall_line[1] < mall_line[0]) \
     and (understanding["W11"] > understanding["W8"]) and (lnk_line[3] < lnk_line[0])
print(f"两对象均呈剪刀差：{ok} → 方法论可定稿：望远镜分排期用，内窥镜分还债用" )

## 实验 2：五维雷达对照

三张雷达叠放：MallSenseAI 望远镜（W12）、MallSenseAI 内窥镜（W13）、LnkChat 终值（W11）。看两点：① 复评下探发生在哪两维（CH/TD，测试纪律托住 CH 底）；② 两对象 Code Health 差 1.0 分——"集成信任边界按 Code Health 画"的量化依据。

In [ ]:
labels = ["架构质量\nAQ", "代码健康\nCH", "ADR一致性\nADR", "技术债\nTD(越高越好)", "开发者体验\nDX"]
ang = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist(); ang += ang[:1]

fig, ax = plt.subplots(figsize=(7.5, 7.5), subplot_kw=dict(polar=True))
for d, name, color, ls in [(mall["W12"], "MallSenseAI W12 望远镜 7.05", "tab:red", "--"),
                            (mall["W13"], "MallSenseAI W13 内窥镜 6.80", "tab:red", "-"),
                            (lnkchat["W11"], "LnkChat W11 终值 6.40", "tab:blue", "-")]:
    vals = [d[k] for k in DIMS]; vals += vals[:1]
    ax.plot(ang, vals, ls, color=color, lw=2, label=name)
    if ls == "-": ax.fill(ang, vals, color=color, alpha=0.08)
ax.set_xticks(ang[:-1]); ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(5, 8); ax.set_yticks([5.5, 6, 6.5, 7, 7.5, 8]); ax.tick_params(axis='y', labelsize=8)
ax.set_title("五维雷达：复评下探集中在 CH/TD；两对象 CH 差 1.0 分", fontsize=12, pad=24)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.18), fontsize=9)
fig.tight_layout(); fig.savefig("/root/learning-notebooks/第13周/w13d7_radar_comparison.png", dpi=130)
plt.show()

print(f"复评维度的变化：" + ", ".join(f"{k} {mall['W12'][k]:.1f}→{mall['W13'][k]:.1f}" for k in DIMS))
print(f"Code Health：MallSenseAI {mall['W13']['CH']} vs LnkChat {lnkchat['W11']['CH']}（差 {mall['W13']['CH']-lnkchat['W11']['CH']:.1f}）→ 信任边界依据成立" )

## 实验 3：集成木桶蒙特卡洛

**模型**（D6 演进路线图的工期估计 → 三角分布）：
- 视觉侧就绪（串行）：S1 立尺子 ~ Tri(1,2,4) 周 + S2 出报表 ~ Tri(2,3,5) 周
- 平台侧就绪（串行）：注册表数据化 ~ Tri(2,4,8) 周 + Connector 路由 ~ Tri(1,2,4) 周
- **集成通车（S3）= max(视觉侧, 平台侧)**（木桶效应：两侧独立并行，慢者定窗口）

**待验证判断**（md §3.2/§3.4）：① 通车窗口落在两侧预期之后；② 长尾由较慢一侧的悲观情形主导；③ "帮慢的一侧"是唯一有效加速——把快侧提速 20% 几乎无效，把慢侧提速 20% 收益显著。

In [ ]:
N = 30_000
def side(n):
    """返回 (视觉侧, 平台侧) 各 n 次抽样的就绪周数"""
    vision = rng.triangular(1, 2, 4, n) + rng.triangular(2, 3, 5, n)      # S1 → S2 串行
    platform = rng.triangular(2, 4, 8, n) + rng.triangular(1, 2, 4, n)    # 注册表 → Connector 串行
    return vision, platform

vision, platform = side(N)
gate = np.maximum(vision, platform)          # 集成通车 = 木桶
slow_is_platform = (platform >= vision)      # 每次抽样谁是短板

print(f"期望工期：视觉侧 {vision.mean():.1f} 周 | 平台侧 {platform.mean():.1f} 周")
print(f"集成通车窗口：中位数 {np.median(gate):.1f} 周 | P90 {np.percentile(gate,90):.1f} 周")
print(f"比慢侧期望多等：{np.median(gate) - max(vision.mean(), platform.mean()):.1f} 周（慢侧自身的波动也要等）")
print(f"长尾来源：P90 情形中 {slow_is_platform[(gate > np.percentile(gate,90))].mean()*100:.0f}% 是平台侧拖的 → 长尾由慢侧悲观情形主导")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))

ax = axes[0]
ax.hist(vision, bins=80, alpha=0.55, color="tab:red", label=f"视觉侧（均值 {vision.mean():.1f} 周）")
ax.hist(platform, bins=80, alpha=0.55, color="tab:blue", label=f"平台侧（均值 {platform.mean():.1f} 周）")
ax.hist(gate, bins=80, histtype="step", lw=2.2, color="black", label=f"通车窗口 = max（中位 {np.median(gate):.1f} 周）")
ax.axvline(np.median(gate), color="black", ls=":", lw=1.2)
ax.set_xlabel("周"); ax.set_ylabel("频数")
ax.set_title("集成窗口分布：形状由慢侧决定", fontsize=12)
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 敏感度：给某一侧整体提速 20%（分布 ×0.8），看通车中位数变化
ax = axes[1]
base = np.median(gate)
def gate_with(scale_v, scale_p, n=N):
    v = (rng.triangular(1,2,4,n) + rng.triangular(2,3,5,n)) * scale_v
    p = (rng.triangular(2,4,8,n) + rng.triangular(1,2,4,n)) * scale_p
    return np.median(np.maximum(v, p))
scenarios = [(1.0, 1.0), (0.8, 1.0), (1.0, 0.8), (0.8, 0.8)]
meds = [gate_with(*s) for s in scenarios]
names = ["基线", "快侧(视觉)\n提速20%", "慢侧(平台)\n提速20%", "双侧\n提速20%"]
colors = ["gray", "tab:red", "tab:blue", "tab:green"]
bars = ax.bar(names, meds, color=colors, alpha=0.75)
for b, m in zip(bars, meds):
    ax.text(b.get_x()+b.get_width()/2, m+0.06, f"{m:.1f}", ha="center", fontsize=10)
ax.axhline(base, color="gray", ls=":", lw=1.2)
ax.set_ylabel("通车窗口中位数（周）")
ax.set_title(f"敏感度：帮快侧仅省 {base-meds[1]:.1f} 周，帮慢侧省 {base-meds[2]:.1f} 周", fontsize=12)
ax.grid(alpha=0.3, axis="y")
fig.tight_layout(); fig.savefig("/root/learning-notebooks/第13周/w13d7_integration_barrel.png", dpi=130)
plt.show()

print(f"结论：帮慢侧收益/帮快侧收益 = {(base-meds[2])/max(base-meds[1],1e-9):.1f}× → 集成加速资源应全部投给短板一侧" )

## 结论汇总

| md 中的判断 | notebook 验证 | 结果 |
|---|---|---|
| 复评 6.8 落在预测区间 6.3-6.8 上沿 | 实验 0 自算综合分 | ✅ 命中 |
| 剪刀差跨对象同构，可定稿为方法论 | 实验 1 双对象曲线 | ✅ 两对象均呈"理解↑评分↓" |
| 复评下探集中在 CH/TD；信任边界按 Code Health 画（差 1.0 分） | 实验 2 雷达对照 | ✅ CH 7.0 vs 6.0 |
| 集成窗口由慢的一侧决定；帮慢侧是唯一有效加速 | 实验 3 蒙特卡洛 + 敏感度 | ✅ 帮慢侧收益数倍于帮快侧 |

**对 W14 开发期的可操作含义**：两侧 Sprint 里，视觉侧 S1（立尺子）和平台侧注册表数据化谁慢谁吃资源；在敏感度结论出来之前，任何"顺手优化快侧"的投入对通车日期都是无效功。